In [29]:
#!/usr/bin/env python3

import requests

def get_best_move(fen: str) -> str:
    """
    Given a FEN string, request the best move from the Chess Cloud Database API.
    
    Returns:
        A move string (e.g. 'e2e4'), or an error message if not found.
    """
    base_url = "http://www.chessdb.cn/cdb.php"
    params = {
        "action": "queryall",
        "board": fen,
        # Optional parameters you might want to add:
        # "endgame": 0,
        # "egtbmetric": "dtz",
        # "learn": 1,
    }

    try:
        response = requests.get(base_url, params=params, timeout=10)
        response_text = response.text.strip()
        print(response_text)
        # Check for known error/edge-case responses:
        if response_text == "invalid board":
            return "Error: The FEN is invalid."
        if response_text == "nobestmove":
            return "Error: No best move found for this position."
        if response_text == "unknown":
            return "Error: Position is unknown to the database."
        if response_text in ["checkmate", "stalemate"]:
            return f"Position result: {response_text}."

        # Otherwise, the response might look like "move:e2e4"
        # or "egtb:d7d8=Q" or "search:e2e4|search:e2e3" etc.
        # We can look for the first recognized move tag:
        for tag in ["move:", "egtb:", "search:"]:
            if tag in response_text:
                # Example: "move:e2e4|search:e2e3|..."
                # Split by '|', find the first chunk with that tag
                chunks = response_text.split("|")
                for chunk in chunks:
                    chunk = chunk.strip()
                    if chunk.startswith(tag):
                        return chunk.split(tag)[1]  # e.g. "e2e4"
        
        # If we reach here, we didn't recognize any standard tag
        return f"Unexpected response: {response_text}"

    except requests.RequestException as e:
        return f"Network error: {str(e)}"

def main():
    # Example usage with a test FEN
    # Starting position FEN:
    # test_fen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
    test_fen = "r3kbr1/3bqpp1/ppn1pn1p/2p5/2P5/PP1QPN1P/1B1PBPP1/3RK2R b Kq - 0 1"

    best_move = get_best_move(test_fen)
    print("Best move:", best_move)

if __name__ == "__main__":
    main()


unknown 
Best move: Unexpected response: unknown 
